# Exercise solutions — Data augmentation and regularization

**DeepVision Masterclass · Solutions companion**

These are worked reference solutions for the exercises in the matching numbered lesson. Run the parent lesson first when variables or datasets are reused. Dataset-specific experiments are left as clearly marked tasks; no performance scores are fabricated.

## Exercise 1 — Compare augmentation policies
Use deterministic validation/test transforms and the same training subset.

In [ ]:
from torchvision import transforms
basic=transforms.Compose([transforms.ToTensor()])
light=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor()])
strong=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ColorJitter(.2,.2,.2,.05),transforms.ToTensor()])
# Run each policy across >=3 seeds and compare validation macro-F1, not only accuracy.

## Exercise 2 — Label smoothing
Use the same model/optimizer but replace the loss.

In [ ]:
import torch
loss_standard=torch.nn.CrossEntropyLoss()
loss_smoothed=torch.nn.CrossEntropyLoss(label_smoothing=.1)
# Tune smoothing in {0.0, 0.05, 0.1, 0.15} on validation data.

## Exercise 3 — MixUp and CutMix
MixUp interpolates images and labels. CutMix pastes a rectangular region and weights labels by the retained area.

In [ ]:
import numpy as np, torch
def mixup_batch(x,y,alpha=.2):
    lam=np.random.beta(alpha,alpha); idx=torch.randperm(x.size(0),device=x.device)
    return lam*x+(1-lam)*x[idx], y, y[idx], lam
def mixup_criterion(logits,ya,yb,lam,criterion):
    return lam*criterion(logits,ya)+(1-lam)*criterion(logits,yb)
def cutmix_batch(x,y,alpha=1.0):
    lam=np.random.beta(alpha,alpha); idx=torch.randperm(x.size(0),device=x.device)
    _,_,h,w=x.shape; cut=np.sqrt(1-lam); ch,cw=int(h*cut),int(w*cut)
    cy,cx=np.random.randint(h),np.random.randint(w)
    y1,y2=max(cy-ch//2,0),min(cy+ch//2,h); x1,x2=max(cx-cw//2,0),min(cx+cw//2,w)
    mixed=x.clone(); mixed[:,:,y1:y2,x1:x2]=x[idx,:,y1:y2,x1:x2]
    lam=1-((y2-y1)*(x2-x1)/(h*w))
    return mixed,y,y[idx],lam

## Exercise 4 — Domain validity
- Digits: small translations/rotations may be valid; large rotations can change digit identity (6/9).
- Traffic signs: horizontal flips can reverse asymmetric symbols and are usually unsafe.
- Retinal fundus: flips/rotations may be plausible but must be clinically validated; avoid color jitter that alters diagnostic cues.
- Satellite imagery: rotations/flips often valid for orientation-invariant land-cover classes, but not for directional tasks (roads/flow).

## Reporting template
Record the dataset/source and license, split strategy, seed(s), preprocessing, model configuration, optimizer, epochs, best validation checkpoint, final test metrics, hardware and runtime. Replace illustrative code comments with measured results from your own run.